In [ ]:
import os

# ====== Runtime parameters (edit here if needed) ======
MODEL_NAME = os.getenv("MODEL_NAME", "Qwen/Qwen3-0.6B")
DATA_FILE = os.getenv("DATA_FILE", "../data/tcm_ancient_modern_grpo_train_12145.json")
OUTPUT_DIR = os.getenv("OUTPUT_DIR", "../outputs/qwen3_0_6b_comet_grpo")
USE_SWANLAB = os.getenv("USE_SWANLAB", "0") == "1"

os.environ["CUDA_VISIBLE_DEVICES"] = os.getenv("CUDA_VISIBLE_DEVICES", "0")
os.environ.setdefault("HF_ENDPOINT", "https://hf-mirror.com")

print("MODEL_NAME =", MODEL_NAME)
print("DATA_FILE  =", DATA_FILE)
print("OUTPUT_DIR =", OUTPUT_DIR)
print("USE_SWANLAB =", USE_SWANLAB)


In [ ]:
import re
import unsloth
from collections import Counter
import torch
from datasets import load_dataset, Dataset
from transformers import AutoTokenizer, AutoModelForCausalLM
from peft import LoraConfig, get_peft_model, TaskType
from unsloth import FastLanguageModel, PatchFastRL
PatchFastRL("GRPO", FastLanguageModel)

In [ ]:
import os
####################################################################################################
# 1.加载模型
# 使用 unsloth 优化的 FastLanguageModel 加载模型
####################################################################################################
# 1.加载模型
# 使用 unsloth 优化的 FastLanguageModel 加载模型
from unsloth import FastLanguageModel
max_seq_length = 4096 # 最大序列长度
dtype = None          # 数据类型，None表示自动选择
load_in_4bit = False  # 使用4bit量化加载模型以节省显存
lora_rank = 32
# 加载预训练模型和分词器
model, tokenizer = FastLanguageModel.from_pretrained(
    model_name = MODEL_NAME,
    max_seq_length = max_seq_length,
    fast_inference = True, # Enable vLLM fast inference
    dtype = dtype,
    temperature=0.95,
    max_lora_rank = lora_rank,
    load_in_4bit = load_in_4bit,
)
print(model)

In [ ]:
model = FastLanguageModel.get_peft_model(
    model,
    r = lora_rank, # Choose any number > 0 ! Suggested 8, 16, 32, 64, 128
    target_modules = [
        "q_proj", "k_proj", "v_proj", "o_proj",
        "gate_proj", "up_proj", "down_proj",
    ],
    lora_alpha = lora_rank*2, # *2 speeds up training
    use_gradient_checkpointing = "unsloth", # Reduces memory usage
    random_state = 3407,
)

In [ ]:
import re
def extract_answer(response):
    if not response:
        return ""
    
    # 先去掉 <think>...</think> 的内容（非贪婪匹配，支持跨行）
    response_no_think = re.sub(r'<think>.*?</think>', '', response, flags=re.DOTALL)
    
    # 提取第一个 <answer>...</answer> 的内容
    match = re.search(r'<answer>(.*?)</answer>', response_no_think, flags=re.DOTALL)
    if match:
        return match.group(1).strip()
    return ""

In [ ]:
from datasets import load_dataset, Dataset

prompt_sys ="""
【角色与背景定位】 你是一位精通中医古籍与古汉语的专家，擅长深度解读医学经典，具备训诂学、临床中医知识及古今术语转换能力。【总体任务要求】 当用户提供中医古籍原文时，你的核心任务是将其准确、流畅地翻译成现代汉语。你应运用下列详细分析步骤进行内部思考和处理，最终目标是生成并仅输出该原文的现代汉语译文。翻译需力求精准，保留原文的理论深度与文化内涵。 【详细分析步骤及每步目的】 1. [原文]内容：列出用户提供的古籍原文。目的：确立分析的文本基础。 2. [术语解析]内容：解析核心术语的字源、本义、在特定古籍或中医理论中的含义、古今语义差异及潜在误读。目的：确保对术语的理解准确且尊重原义，为后续分析奠定词义基础。 3. [医理推演]内容：结合中医基础理论（阴阳五行、脏腑经络、气血津液、病因病机等）及所分析古籍的理论，推演原文涉及的生理、病理机制。可参考现代医学知识，但以中医理论为主体。目的：构建文本背后的医学知识模型，确保分析既“通顺”又“有理”。 4. [句法解构]内容：分析古文句式、特殊语法现象（倒装、互文、省略等），拆解复杂结构，补全省略成分，明确句子结构。目的：还原古文句法原貌，为准确理解和转换为现代汉语扫清障碍。 5. [文化注释]内容：解读原文体现的中医哲学思想（如天人相应、整体观）、文化背景、象征比喻及其相关的宇宙观、生命观。目的：传递中医经典的文化精髓与哲学智慧。 6. [现代译文]内容：陈述用户提供的现代汉语译文。目的：明确分析和推理所参照的目标译文。【输入】 用户将提供 {古文原文}。【输出格式】 严格按照以下格式输出。 {用户提供的古文原文} 为用户输入。你需要生成 <think> 标签内的所有分析内容，并确保 <answer> 标签内为你生成的现代汉语译文。 <think> [原文] {用户提供的古文原文} [术语解析] {核心术语的训诂、辨析、在特定古籍中的含义、古今语义差异及易混淆点。} [医理推演] {基于中医理论对原文生理、病理机制的深度阐发与论证。} [句法解构] {古文句式结构、特殊语法现象分析，还原句意。} [文化注释] {原文的中医哲学、文化背景、象征意义及相关宇宙生命观解读。} [现代译文] {你生成的现代汉语译文} </think> <answer>{你生成的现代汉语译文}</answer> 您的回复：
"""
# 加载 JSON 数据
dataset = load_dataset("json", data_files=DATA_FILE)
dataset1 = dataset["train"]  # 默认是 train split

# 转换为列表格式，每个元素是 dict
dataset_list = dataset1.to_list()

# 构造 prompt-answer 格式的数据
def process_data(data_list):
    processed = []
    for sample in data_list:
        query = sample.get('ancient', '').strip()
        answer = sample.get('modern', '').strip()
        if not query or not answer:
            continue  # 跳过缺失项
        item = {
            'prompt': [
                {'role': 'system', 'content': prompt_sys.replace('\n','')},
                {'role': 'user', 'content': query}
            ],
            'answer': answer
        }
        processed.append(item)
    return processed

# 处理数据
processed_data = process_data(dataset_list)
processed_dataset = Dataset.from_list(processed_data)

# 打印第 3 条（从 0 开始，第 4 条）
print(processed_data[3])# 应用 tokenizer（假设你已有 tokenizer）
tokenized = processed_dataset.map(
    lambda x: {
        "tokens": tokenizer.apply_chat_template(
            x["prompt"], add_generation_prompt=True, tokenize=True
        )
    }
)
# 看一下结果
print(tokenizer.decode(tokenized[0]["tokens"]))

In [ ]:
from comet import download_model, load_from_checkpoint

# 1️⃣ 下载并加载 COMET 模型
model_path = download_model("Unbabel/wmt22-comet-da")
model_comet = load_from_checkpoint(model_path)
def cal_comet_reward_batch(data_list, batch_size=32, gpus=1):
    """
    批量计算 COMET 奖励（并行优化版）
    
    Args:
        data_list: List of dicts with keys 'src', 'mt', 'ref'
        batch_size: 批处理大小，根据显存调整（默认32）
        gpus: GPU数量（默认1，设为0使用CPU）
    
    Returns:
        List of scores
    """
    if not data_list:
        return []
    
    # ✅ 一次性批量预测所有数据
    output = model_comet.predict(data_list, batch_size=batch_size, gpus=gpus)
    
    # 返回每个样本的分数
    scores = [round(float(score), 2) for score in output.scores]
    return scores

In [ ]:
def comet_reward_func(prompts, completions, answer, batch_size=32, gpus=1, **kwargs):
    """
    并行计算 COMET 奖励（优化版）
    
    Args:
        prompts: 提示列表
        completions: 生成结果列表
        answer: 参考答案列表
        batch_size: COMET 模型批处理大小（默认32，可根据显存调整）
        gpus: GPU数量（默认1）
    
    Returns:
        all_rewards: 所有样本的奖励分数列表
    """
    # 第一步：收集所有需要计算的数据
    data_list = []
    index_mapping = []  # 记录每个数据对应的原始索引 (i, j)
    
    for i, gens in enumerate(completions):
        ref_answer = answer[i]
        responses = [gen['content'] for gen in gens]
        extracted_responses = [extract_answer(r) for r in responses]
        src_text = prompts[i][-1]['content']
        
        # 收集当前样本的所有候选答案
        for j, extracted_response in enumerate(extracted_responses):
            data_list.append({
                "src": src_text,
                "mt": extracted_response,
                "ref": ref_answer
            })
            index_mapping.append((i, j))
    
    # 第二步：批量并行计算所有 COMET 分数
    print(f"\n🔄 开始批量计算 {len(data_list)} 个样本的 COMET 奖励...")
    print(f"   批处理大小: {batch_size}, GPU数量: {gpus}")
    all_rewards = cal_comet_reward_batch(data_list, batch_size=batch_size, gpus=gpus)
    print(f"✅ 批量计算完成！平均奖励: {sum(all_rewards)/len(all_rewards):.4f}\n")
    return all_rewards

In [ ]:
import sacrebleu
from sacrebleu.metrics import BLEU, CHRF
import jieba

# 使用 jieba 进行中文分词
def tokenize_chinese(text):
    # 确保输入是字符串
    text = str(text)
    # 使用 jieba 进行分词，然后用空格连接
    return " ".join(jieba.cut(text))
bleu_metric = BLEU(tokenize="none")
chrf_metric = CHRF()

def cal_blue_reward(completion, answer, **kwargs):
    # 使用 jieba 分词后的文本
    sys_translations = [tokenize_chinese(completion)]
    refs_translations = [[tokenize_chinese(answer)]]
    
    # 直接获取分数，范围 0-100
    bleu_score = bleu_metric.corpus_score(sys_translations, refs_translations).score
    bleu_reward = (max(0, bleu_score) / 100.0)

    reward =  bleu_reward
    return round(reward, 2)

In [ ]:
import os
from datetime import datetime

# 假设你的 cal_blue_reward 和 extract_answer 函数已经定义好了
# from your_module import cal_blue_reward, extract_answer

def bleu_reward_func(prompts, completions, answer, file_path='bleu_rewards_8.25.log', **kwargs):
    """
    计算 BLEU 奖励并将其详细过程保存到文件中。

    参数:
    prompts (list): 包含问题的列表。
    completions (list): 包含模型生成回答的列表。
    answer (list): 包含参考答案的列表。
    file_path (str): 日志文件的保存路径，默认为 'bleu_rewards.log'。
    **kwargs: 传递给 cal_blue_reward 函数的额外参数。

    返回:
    list: 所有生成回答的 BLEU 奖励分数列表。
    """
    all_rewards = []
    
    # 使用 'a' 模式打开文件，表示追加写入，这样每次调用都会在文件末尾添加内容
    with open(file_path, 'a', encoding='utf-8') as f:
        # 写入本次调用的时间戳，方便追溯
        timestamp = datetime.now().strftime("%Y-%m-%d %H:%M:%S")
        f.write(f"\n{'='*20} 奖励计算开始于 {timestamp} {'='*20}\n")
        for i, gens in enumerate(completions):
            ref_answer = answer[i]
            responses = [gen['content'] for gen in gens]
            extracted_responses = [extract_answer(r) for r in responses]
            # 遍历每个生成的回答并计算奖励
            for j, extracted_response in enumerate(extracted_responses):
                reward = cal_blue_reward(extracted_response, ref_answer, **kwargs)
            
    return all_rewards

In [ ]:
from typing import List, Dict
# 统一的格式奖励函数
THINK_PATTERN = re.compile(r"<think>(.*?)</think>", re.DOTALL)
ANSWER_PATTERN = re.compile(r"<answer>(.*?)</answer>", re.DOTALL)
ORDER_PATTERN = re.compile(r"<think>.*?</think>\s*<answer>.*?</answer>", re.DOTALL)
NORMALIZE_PATTERN = re.compile(r'[^\u4e00-\u9fff\w]')

def format_reward_func(completions: List[List[Dict]], **kwargs) -> List[float]:
    """优化后的格式奖励函数，使用预编译正则和向量化计算"""
    all_rewards = []
    
    # 🚀 批量处理所有回答
    all_responses = []
    for gens in completions:
        for gen in gens:
            all_responses.append(gen["content"])
    
    # 🚀 向量化格式检查
    rewards = []
    for response in all_responses:
        reward = 0.0
        
        # 使用预编译的正则表达式
        think_match = THINK_PATTERN.search(response)
        answer_match = ANSWER_PATTERN.search(response)
        
        # 基础标签存在性奖励
        if think_match:
            reward += 0.1
        if answer_match:
            reward += 0.1
        
        # 格式惩罚检查（优化版）
        if think_match and answer_match:
            content_to_check = response
            content_to_check = content_to_check.replace(think_match.group(0), "", 1)
            content_to_check = content_to_check.replace(answer_match.group(0), "", 1)
            
            if content_to_check.strip():
                reward -= 0.5
            
            # 结构顺序检查
            if ORDER_PATTERN.search(response):
                reward += 0.2
            
            # think内容结构检查（简化版）
            think_content = think_match.group(1)
            required_sections = ["原文", "术语解析", "医理推演", "句法解构", "文化注释", "现代译文"]
            
            found_count = sum(1 for section in required_sections if section in think_content)
            reward += 0.05 * found_count
            
            if found_count > 0:
                # 简化的顺序检查
                last_pos = -1
                is_ordered = True
                for section in required_sections:
                    pos = think_content.find(section)
                    if pos != -1:
                        if pos < last_pos:
                            is_ordered = False
                            break
                        last_pos = pos
                
                if is_ordered:
                    reward += 0.3
        
        rewards.append(round(reward, 4))
    
    return rewards

In [ ]:
import unsloth
from trl import GRPOConfig, GRPOTrainer

callbacks = []
if USE_SWANLAB:
    from swanlab.integration.transformers import SwanLabCallback
    swanlab_callback = SwanLabCallback(
        project="rl",
        experiment_name="qwen3_grpo",
        description="GRPO experiment for TCM translation",
        config={"framework": "unsloth", "algorithm": "GRPO"},
    )
    callbacks.append(swanlab_callback)


In [ ]:
from unsloth import is_bfloat16_supported

num_generations = 8  # 从8增加到12，增加生成数量
num_prompts_per_device = 4 # 从4增加到6，增加每设备提示数量
output_dir = OUTPUT_DIR

training_args = GRPOConfig(
    # use_vllm = True,  # 启用vLLM加速生成，用空间换时间
    learning_rate = 5e-5,
    weight_decay = 0.1,
    warmup_ratio = 0.01,
    temperature = 0.95,
    lr_scheduler_type = "cosine",
    optim = "adamw_8bit",
    logging_steps = 1,
    per_device_train_batch_size = num_generations*num_prompts_per_device,  
    gradient_accumulation_steps = 1,  # 从2减少到1，减少梯度累积时间
    num_generations = num_generations,  # Decrease if out of memory
    max_prompt_length = 1500,
    max_completion_length = 2596,
    num_train_epochs = 2, # Set to 1 for a full training run
    # max_steps = 800,
    save_steps = 200,
    max_grad_norm = 0.2,
    report_to = "tensorboard",  # Can use Weights & Biases
    output_dir = output_dir,
    log_on_each_node=True,
) 

In [ ]:
trainer = GRPOTrainer(
model=model,
processing_class=tokenizer,
reward_funcs=[
   # cosine_reward_func,
    comet_reward_func,
              bleu_reward_func,
              format_reward_func,
    # mixed_reward_function
             ],  # 使用整合后的奖励函数
args=training_args,
train_dataset=processed_data,
     callbacks=callbacks,
    # use_gradient_checkpointing=True
)

trainer.train()
trainer.save_model(output_dir)